In [86]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import optuna
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import accuracy_score
from optuna.visualization import plot_terminator_improvement
from optuna.terminator import report_cross_validation_scores
import json
import matplotlib.pyplot as plt
from tabulate import tabulate

In [87]:
X_train= pd.read_csv('/kaggle/input/datasets/prahazra/employee-data-processed/X_train_final.csv')
x_test= pd.read_csv('/kaggle/input/datasets/prahazra/employee-data-processed/X_test_final.csv')

Y_train= pd.read_csv('/kaggle/input/datasets/prahazra/employee-data-processed/y_train_final.csv').squeeze()
y_test= pd.read_csv('/kaggle/input/datasets/prahazra/employee-data-processed/y_test_final.csv')

In [88]:
print(Y_train.value_counts())
print(len(X_train))

Attrition
0    986
1    905
Name: count, dtype: int64
1891


In [89]:
x_test.drop(columns=['Unnamed: 0'], inplace= True)
y_test.drop(columns=['Unnamed: 0'], inplace= True)

In [90]:
if hasattr(y_test, 'squeeze'):
    y_test = y_test.squeeze()

In [91]:
y_test

0      0
1      0
2      1
3      1
4      0
      ..
289    0
290    0
291    0
292    0
293    0
Name: Attrition, Length: 294, dtype: int64

# Performance Before Optimization

In [92]:
LR= LogisticRegression(random_state= 0)
RF= RandomForestClassifier(random_state = 0)
GB= GradientBoostingClassifier(random_state= 0)
XGB= XGBClassifier(random_state= 0)

In [93]:
LR.fit(X_train, Y_train)
RF.fit(X_train, Y_train)
GB.fit(X_train, Y_train)
XGB.fit(X_train, Y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [94]:
Models= {
    "Logistic Regression": LR,
    "Random Forest": RF,
    "Gradient Boosting": GB,
    "XGBoost": XGB
}

performance_BO= []
for name, model in Models.items():
    y_pred= model.predict(x_test)
    accuracy = accuracy_score(y_pred, y_test)
    performance_BO.append([name, f"{accuracy * 100:.2f}%"])

headers= ['Model', 'Test Accuracy']
print(tabulate(performance_BO, headers= headers, tablefmt= 'grid'))

+---------------------+-----------------+
| Model               | Test Accuracy   |
+=====================+=================+
| Logistic Regression | 75.17%          |
+---------------------+-----------------+
| Random Forest       | 85.37%          |
+---------------------+-----------------+
| Gradient Boosting   | 87.41%          |
+---------------------+-----------------+
| XGBoost             | 86.39%          |
+---------------------+-----------------+


# Model Optimization using Optuna

In [95]:
sampler= optuna.samplers.TPESampler(seed =0)

In [96]:
def objective_LR(trial):
    params= {
        'penalty' : trial.suggest_categorical('penalty', ['l1', 'l2', 'elasticnet']),
        'C' : trial.suggest_float('C', 1e-5, 10.0, log=True),
        'tol' : trial.suggest_float('tol', 1e-6, 1e-1, log= True),
        'solver' : trial.suggest_categorical('solver', ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga']),
        'max_iter' : trial.suggest_int('max_iter', 50 , 200)
    }
    if params['penalty'] == 'elasticnet':
        params['l1_ratio'] = trial.suggest_float('l1_ratio', 0 ,1)
    
    try:
        model= LogisticRegression(**params, random_state = 0, n_jobs = -1)
        scores = cross_val_score(model, X_train, Y_train, 
                                 cv=KFold(n_splits=5, shuffle=True, random_state= 0))
        
        report_cross_validation_scores(trial, scores)
        return np.mean(scores)
    except Exception as e:
        raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [97]:
study_LR = optuna.create_study(direction="maximize", study_name= "Optimize_LR", sampler= sampler)
study_LR.optimize(objective_LR, n_trials=50, show_progress_bar=True)
fig_LR= plot_terminator_improvement(study_LR, plot_error = True)
fig_LR.show()

[I 2026-06-10 03:06:56,041] A new study created in memory with name: Optimize_LR


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-10 03:07:01,975] Trial 0 finished with value: 0.7789546425430331 and parameters: {'penalty': 'l2', 'C': 0.01859084363016962, 'tol': 0.0001313028028065861, 'solver': 'newton-cholesky', 'max_iter': 129}. Best is trial 0 with value: 0.7789546425430331.
[I 2026-06-10 03:07:02,072] Trial 1 finished with value: 0.5240594156161438 and parameters: {'penalty': 'l2', 'C': 3.332543279005119e-05, 'tol': 1.2620948285169311e-06, 'solver': 'newton-cholesky', 'max_iter': 167}. Best is trial 0 with value: 0.7789546425430331.
[I 2026-06-10 03:07:02,226] Trial 2 finished with value: 0.8016850246401697 and parameters: {'penalty': 'l2', 'C': 4.656005689076002, 'tol': 0.0004066695065393846, 'solver': 'newton-cg', 'max_iter': 143}. Best is trial 2 with value: 0.8016850246401697.
[I 2026-06-10 03:07:02,250] Trial 3 pruned. Pruned due to runtime error: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1271: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  

[I 2026-06-10 03:07:02,934] Trial 6 finished with value: 0.7350588432382628 and parameters: {'penalty': 'l2', 'C': 0.0004975557797102343, 'tol': 3.9900910508086484e-06, 'solver': 'saga', 'max_iter': 135}. Best is trial 2 with value: 0.8016850246401697.
[I 2026-06-10 03:07:03,026] Trial 7 finished with value: 0.7858273652468902 and parameters: {'penalty': 'l2', 'C': 0.028554790180743944, 'tol': 0.04430788173937543, 'solver': 'newton-cholesky', 'max_iter': 138}. Best is trial 2 with value: 0.8016850246401697.
[I 2026-06-10 03:07:03,133] Trial 8 finished with value: 0.7926945037762979 and parameters: {'penalty': 'l2', 'C': 0.11665388871103255, 'tol': 2.2389266508862164e-05, 'solver': 'liblinear', 'max_iter': 83}. Best is trial 2 with value: 0.8016850246401697.
[I 2026-06-10 03:07:03,154] Trial 9 pruned. Pruned due to runtime error: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more deta

...

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[I 2026-06-10 03:07:11,075] Trial 46 finished with value: 0.7963968114363892 and parameters: {'penalty': 'l1', 'C': 0.7848050363473787, 'tol': 0.00018308494067647793, 'solver': 'saga', 'max_iter': 122}. Best is trial 31 with value: 0.8027418296547584.
[I 2026-06-10 03:07:11,191] Trial 47 finished with value: 0.7995714146109923 and parameters: {'penalty': 'l2', 'C': 2.243120193183524, 'tol': 2.895006225094525e-05, 'solver': 'newton-cholesky', 'max_iter': 153}. Best is trial 31 with value: 0.8027418296547584.


/tmp/ipykernel_58/2950119267.py:3: ExperimentalWarning: optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.
  fig_LR= plot_terminator_improvement(study_LR, plot_error = True)
/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:93: ExperimentalWarning: RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.
  improvement_evaluator = RegretBoundEvaluator()
/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:98: ExperimentalWarning: CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.
  error_evaluator = CrossValidationErrorEvaluator()


[I 2026-06-10 03:07:11,296] Trial 48 finished with value: 0.774195529868353 and parameters: {'penalty': 'l2', 'C': 0.012011254410157777, 'tol': 0.0018024405921325806, 'solver': 'lbfgs', 'max_iter': 106}. Best is trial 31 with value: 0.8027418296547584.
[I 2026-06-10 03:07:11,322] Trial 49 pruned. Pruned due to runtime error: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^

100%|██████████| 50/50 [00:08<00:00,  5.68it/s]


In [98]:
def objective_RF(trial):
    params= {
        'n_estimators' : trial.suggest_int('n_estimators', 100, 300),
        'criterion' : trial.suggest_categorical('criterion', ['gini','entropy','log_loss']),
        'max_depth' : trial.suggest_int('max_depth', 5 ,10),
        'min_samples_split' : trial.suggest_int('min_samples_split', 10, 40),
        'bootstrap' : trial.suggest_categorical('bootstrap', [True, False]),
        'class_weight': trial.suggest_categorical('class_weight', ['balanced', 'balanced_subsample', None])
    }
    max_features_option = trial.suggest_categorical('max_features_option', ['sqrt', 'log2', None, 'float']) # Corrected parameter name
    if max_features_option == 'float':
        params['max_features'] = trial.suggest_float('max_features', 0.1, 1.0)
    else:
        params['max_features'] = max_features_option
    if params['bootstrap'] == True:
        params['max_samples']= trial.suggest_float('max_samples', 0.5, 0.9)

    try:
        model= RandomForestClassifier(**params, random_state = 0, n_jobs = -1)
        scores = cross_val_score(model, X_train, Y_train, 
                                 cv=KFold(n_splits=5, shuffle=True, random_state= 0))
        
        report_cross_validation_scores(trial, scores)
        return np.mean(scores)
    except Exception as e:
        raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [99]:
study_RF = optuna.create_study(direction="maximize", study_name= "Optimize_RF", sampler= sampler)
study_RF.optimize(objective_RF, n_trials=50, show_progress_bar=True)
fig_RF= plot_terminator_improvement(study_RF, plot_error = True)
fig_RF.show()

[I 2026-06-10 03:07:23,454] A new study created in memory with name: Optimize_RF


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-10 03:07:33,996] Trial 0 finished with value: 0.9169730982395891 and parameters: {'n_estimators': 200, 'criterion': 'gini', 'max_depth': 8, 'min_samples_split': 10, 'bootstrap': False, 'class_weight': 'balanced_subsample', 'max_features_option': 'float', 'max_features': 0.6168927239646209}. Best is trial 0 with value: 0.9169730982395891.
[I 2026-06-10 03:07:37,862] Trial 1 finished with value: 0.9100933953176698 and parameters: {'n_estimators': 231, 'criterion': 'log_loss', 'max_depth': 7, 'min_samples_split': 23, 'bootstrap': True, 'class_weight': None, 'max_features_option': 'log2', 'max_samples': 0.5649971738705499}. Best is trial 0 with value: 0.9169730982395891.
[I 2026-06-10 03:07:42,522] Trial 2 finished with value: 0.9243735254289345 and parameters: {'n_estimators': 223, 'criterion': 'entropy', 'max_depth': 8, 'min_samples_split': 22, 'bootstrap': False, 'class_weight': None, 'max_features_option': 'sqrt'}. Best is trial 2 with value: 0.9243735254289345.
[I 2026-06-1

/tmp/ipykernel_58/2767116058.py:3: ExperimentalWarning:

optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:93: ExperimentalWarning:

RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:98: ExperimentalWarning:

CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.



[I 2026-06-10 03:11:36,389] Trial 49 finished with value: 0.9386480713657495 and parameters: {'n_estimators': 283, 'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 10, 'bootstrap': False, 'class_weight': 'balanced_subsample', 'max_features_option': 'log2'}. Best is trial 49 with value: 0.9386480713657495.


100%|██████████| 50/50 [00:05<00:00,  9.04it/s]


In [100]:
def objective_GB(trial):
    params = {
        'loss': trial.suggest_categorical('loss', ['log_loss', 'exponential']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'criterion': trial.suggest_categorical('criterion', ['friedman_mse', 'squared_error']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_impurity_decrease': trial.suggest_float('min_impurity_decrease', 0.0, 0.5),
        'max_depth': trial.suggest_int('max_depth', 1, 20),
        'random_state': 0
    }
    max_features_option = trial.suggest_categorical('max_features_option', ['sqrt', 'log2', None, 'float']) 
    if max_features_option == 'float':
        params['max_features'] = trial.suggest_float('max_features_gb_float', 0.1, 1.0)
    else:
        params['max_features'] = max_features_option

    try:
        model = GradientBoostingClassifier(**params)
        scores = cross_val_score(model, X_train, Y_train, cv=KFold(n_splits=5, shuffle=True,random_state= 0))
        report_cross_validation_scores(trial, scores)
        return np.mean(scores)
    except Exception as e:
        raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [101]:
study_GB = optuna.create_study(direction="maximize", study_name= "Optimize_GB", sampler = sampler)
study_GB.optimize(objective_GB, n_trials=50, show_progress_bar=True)
fig_GB= plot_terminator_improvement(study_GB, plot_error = True)
fig_GB.show()

[I 2026-06-10 03:11:43,278] A new study created in memory with name: Optimize_GB


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-06-10 03:11:45,544] Trial 0 finished with value: 0.9312504362636289 and parameters: {'loss': 'log_loss', 'learning_rate': 0.1956272447321156, 'n_estimators': 182, 'criterion': 'friedman_mse', 'min_samples_split': 2, 'min_impurity_decrease': 0.17361675896610979, 'max_depth': 3, 'max_features_option': 'sqrt'}. Best is trial 0 with value: 0.9312504362636289.
[I 2026-06-10 03:11:47,352] Trial 1 finished with value: 0.9243763175161591 and parameters: {'loss': 'log_loss', 'learning_rate': 0.1638202474445442, 'n_estimators': 135, 'criterion': 'friedman_mse', 'min_samples_split': 3, 'min_impurity_decrease': 0.43109575871084166, 'max_depth': 20, 'max_features_option': 'sqrt'}. Best is trial 0 with value: 0.9312504362636289.
[I 2026-06-10 03:11:51,324] Trial 2 finished with value: 0.5214153090142537 and parameters: {'loss': 'exponential', 'learning_rate': 0.02203119160109112, 'n_estimators': 109, 'criterion': 'squared_error', 'min_samples_split': 2, 'min_impurity_decrease': 0.38529037425

/tmp/ipykernel_58/3313013327.py:3: ExperimentalWarning:

optuna.visualization._terminator_improvement.plot_terminator_improvement is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:93: ExperimentalWarning:

RegretBoundEvaluator is experimental (supported from v3.2.0). The interface can change in the future.

/usr/local/lib/python3.12/dist-packages/optuna/visualization/_terminator_improvement.py:98: ExperimentalWarning:

CrossValidationErrorEvaluator is experimental (supported from v3.2.0). The interface can change in the future.



[I 2026-06-10 03:20:52,605] Trial 49 finished with value: 0.9249026259580349 and parameters: {'loss': 'log_loss', 'learning_rate': 0.016987120105626072, 'n_estimators': 409, 'criterion': 'friedman_mse', 'min_samples_split': 2, 'min_impurity_decrease': 0.0724763812957607, 'max_depth': 7, 'max_features_option': None}. Best is trial 33 with value: 0.9444667811422429.


100%|██████████| 50/50 [00:05<00:00,  9.64it/s]


In [102]:
def objective_XGB(trial):
    params = {
      'n_estimators': trial.suggest_int('n_estimators', 50, 500),
      'max_depth': trial.suggest_int('max_depth', 1, 20),
      'grow_policy': trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide']), 
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
      'booster': trial.suggest_categorical('booster', ['gbtree', 'gblinear', 'dart']),
      'gamma': trial.suggest_float('gamma', 0.0, 0.5),
      'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
      'use_label_encoder': False, 
      'eval_metric': 'logloss', 
      'random_state': 0
    }
    
    try:
      model = XGBClassifier(**params, n_jobs= -1)
      scores = cross_val_score(model, X_train, Y_train, cv=KFold(n_splits=5, shuffle=True,random_state= 0))
      report_cross_validation_scores(trial, scores)
    
      return np.mean(scores)
    
    except Exception as e:
      raise optuna.TrialPruned(f"Pruned due to runtime error: {e}")

In [103]:
study_XGB = optuna.create_study(direction="maximize", study_name= "Optimize_XGB", sampler= sampler)
study_XGB.optimize(objective_XGB, n_trials=50, show_progress_bar=True)
fig_XGB= plot_terminator_improvement(study_XGB, plot_error = True)
fig_XGB.show()

[I 2026-06-10 03:20:59,094] A new study created in memory with name: Optimize_XGB


  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:20:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:20:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:20:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:21:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:21:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.




[I 2026-06-10 03:21:00,827] Trial 0 finished with value: 0.8937010512208401 and parameters: {'n_estimators': 57, 'max_depth': 9, 'grow_policy': 'lossguide', 'learning_rate': 0.02121687847073361, 'booster': 'gbtree', 'gamma': 0.05774214856937404, 'min_child_weight': 7}. Best is trial 0 with value: 0.8937010512208401.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:21:00] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:21:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:21:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:21:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.




...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:27] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "gamma", "grow_policy", "max_depth", "min_child_weight", "use_label_encoder" } are not used.




[I 2026-06-10 03:31:29,112] Trial 48 finished with value: 0.791105806145384 and parameters: {'n_estimators': 393, 'max_depth': 11, 'grow_policy': 'depthwise', 'learning_rate': 0.05562656676956745, 'booster': 'gblinear', 'gamma': 0.35390028757426184, 'min_child_weight': 3}. Best is trial 20 with value: 0.9397174407728496.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning:

[03:31:31] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.


/tmp/ipykernel_58/2259274934.py:3: 

[I 2026-06-10 03:31:32,545] Trial 49 finished with value: 0.8434644218285378 and parameters: {'n_estimators': 60, 'max_depth': 1, 'grow_policy': 'lossguide', 'learning_rate': 0.06874667077072781, 'booster': 'dart', 'gamma': 0.0412812890114202, 'min_child_weight': 1}. Best is trial 20 with value: 0.9397174407728496.


100%|██████████| 50/50 [00:05<00:00,  8.40it/s]


In [104]:
Model_names= ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'XGBoost']
Model_scores= [study_LR.best_trial.value, study_RF.best_trial.value, study_GB.best_trial.value, study_XGB.best_trial.value]
Model_params= [study_LR.best_trial.params, study_RF.best_trial.params, study_GB.best_trial.params, study_XGB.best_trial.params]
performance= pd.DataFrame({'Model': Model_names, 'Score': Model_scores, 'Parameters': Model_params})
performance[['Model', 'Score']]

,Model,Score
0,Logistic Regression,0.802742
1,Random Forest,0.938648
2,Gradient Boosting,0.944467
3,XGBoost,0.939717


In [105]:
def save_params(prefix_name, params_dict):
    file_name= f"{prefix_name}.json"
    with open(file_name, "w") as f:
        json.dump(params_dict, f, indent= 4)
    print(f"Successfully saved to: {file_name}")

In [106]:
LR_params= study_LR.best_params
save_params("LR_params", LR_params)
RF_params= study_RF.best_params
save_params('RF_params', RF_params)
GB_params= study_GB.best_params
save_params("GB_params", GB_params)
XGB_params= study_XGB.best_params
save_params("XGB_params", XGB_params)

Successfully saved to: LR_params.json
Successfully saved to: RF_params.json
Successfully saved to: GB_params.json
Successfully saved to: XGB_params.json
